# example_training_plots_gpu — GPU walkthrough

Same end-to-end MLflow workflow as `example_training_plots.ipynb`, but the
loss-predictor model is a small **PyTorch MLP trained on CUDA** instead of
an sklearn Ridge on CPU.

**What's different from the CPU notebook**
* `select_device()` picks `cuda` when available, falls back to `cpu` so the
  notebook always runs.
* The predictor is a 2 → 32 → 32 → 1 MLP trained with Adam.
* Features are standardised before training.
* The model is logged with the `pytorch` MLflow flavor (different on-disk
  format than sklearn — state_dict + a CPU-side input example).
* A separate registered name (`burnit-bg-loss-predictor-gpu`) so this run
  doesn't add versions to the CPU model entry.
* Run params include `torch_version`, `cuda_version`, `cuda_device_name`
  for easy CPU-vs-GPU comparison in the UI.


## Naming conventions

Same two-name distinction as the CPU notebook (artifact path vs. registered
model name) — only the values differ so the GPU run goes into its own
registry entry.

In [1]:
MODEL_ARTIFACT_PATH = "loss-predictor-gpu"
REGISTERED_MODEL_NAME = "burnit-bg-loss-predictor-gpu"
MODEL_ALIAS = "staging"
SOURCE_ARTIFACT_PATH = "notebooks"


## Device selection

`torch.cuda.is_available()` returns True only when a CUDA build of torch is
installed *and* the driver/CUDA runtime can find a GPU. The fallback to CPU
keeps the notebook usable on machines without a GPU.

In [2]:
import torch


def select_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


device = select_device()
print("device:", device.type)
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda or "n/a")
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))


device: cuda
torch version: 2.11.0+cu126
cuda available: True
cuda version: 12.6
device name: NVIDIA GeForce RTX 3050 Laptop GPU


## Helpers — same simulator as the CPU notebook

These three helpers are identical to the CPU walkthrough; the only thing
that changes is the *model* fitted on top of the simulated history.

In [3]:
def cosine_lr_schedule(step, total, warmup, peak_lr, min_lr):
    """Linear warmup -> cosine decay. The standard LLM pretraining schedule."""
    import math
    if step < warmup:
        return peak_lr * (step + 1) / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


In [4]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def simulate_training(*, total_steps=600, warmup_steps=60, eval_every=50,
                      peak_lr=3e-4, min_lr=1e-5, seed=42):
    """Generate plausible training and eval metric series.

    Returns a dict of numpy arrays: train/eval loss, perplexity, learning rate,
    grad norms (with occasional spikes), throughput (tokens/sec), and step time.
    """
    rng = np.random.default_rng(seed)
    steps = np.arange(1, total_steps + 1)

    # train loss: exponential decay + small sinusoidal drift + noise, clipped at 1.2
    base = 7.5 * np.exp(-steps / 220.0) + 1.6
    drift = 0.05 * np.sin(steps / 35.0)
    noise = rng.normal(0.0, 0.04, size=total_steps)
    train_loss = np.clip(base + drift + noise, 1.2, None)

    eval_steps = steps[steps % eval_every == 0]
    eval_loss = np.array([train_loss[s - 1] + 0.18 + rng.normal(0.0, 0.03) for s in eval_steps])

    eval_precision, eval_recall, eval_f1, eval_accuracy = [], [], [], []
    for loss in eval_loss:
        n = 1500
        y_true = rng.integers(0, 2, size=n)
        sep = float(np.clip((8.0 - loss) / 8.0, 0.10, 0.90))
        logits = (2 * y_true - 1) * sep + rng.normal(0.0, 0.95, size=n)
        y_pred = (logits > 0.0).astype(int)
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
        eval_precision.append(float(p))
        eval_recall.append(float(r))
        eval_f1.append(float(f))
        eval_accuracy.append(float(accuracy_score(y_true, y_pred)))

    train_ppl = np.exp(train_loss)
    eval_ppl = np.exp(eval_loss)

    lr = np.array([cosine_lr_schedule(s - 1, total_steps, warmup_steps, peak_lr, min_lr)
                   for s in steps])

    grad_norm = np.abs(rng.normal(0.6, 0.18, size=total_steps))
    grad_norm[:warmup_steps] *= 1.6
    spike_idx = rng.choice(total_steps // 4, size=4, replace=False)
    grad_norm[spike_idx] *= rng.uniform(3.5, 5.5, size=spike_idx.size)

    base_tps = 6500.0 + 700.0 * (1.0 - np.exp(-steps / 50.0))
    jitter = rng.normal(0.0, 200.0, size=total_steps)
    tokens_per_sec = np.clip(base_tps + jitter, 3000.0, None)

    batch_tokens = 8192
    step_seconds = batch_tokens / tokens_per_sec

    return {
        "steps": steps, "train_loss": train_loss, "eval_steps": eval_steps,
        "eval_loss": eval_loss, "eval_precision": np.array(eval_precision),
        "eval_recall": np.array(eval_recall), "eval_f1": np.array(eval_f1),
        "eval_accuracy": np.array(eval_accuracy), "train_ppl": train_ppl,
        "eval_ppl": eval_ppl, "learning_rate": lr, "grad_norm": grad_norm,
        "tokens_per_sec": tokens_per_sec, "step_seconds": step_seconds,
    }


In [5]:
def fake_attention_matrix(n_tokens=16, seed=7):
    """Synthetic causal attention pattern with a soft diagonal bias."""
    rng = np.random.default_rng(seed)
    raw = rng.gamma(1.5, 1.0, size=(n_tokens, n_tokens))
    raw = np.tril(raw)
    for i in range(n_tokens):
        raw[i, max(0, i - 2): i + 1] *= 3.0
    raw = np.exp(raw - raw.max(axis=1, keepdims=True))
    raw = raw * np.tri(n_tokens)
    raw = raw / raw.sum(axis=1, keepdims=True).clip(1e-12)
    return raw


## The loss-predictor MLP

A tiny multi-layer perceptron — 2 input features (`step`, `learning_rate`)
through two hidden layers of 32 units, regressing to a single scalar.

For a real pretraining job you'd swap this for the actual model architecture.
For this example the MLP is large enough to demonstrate GPU training (memcpy
to device, forward/backward on CUDA, optimizer step) but small enough to
finish in milliseconds.

In [6]:
from torch import nn


class LossPredictor(nn.Module):
    """2 -> 32 -> 32 -> 1 MLP that predicts train_loss from (step, lr)."""

    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_loss_predictor(x, y, device, *, epochs=400, lr=5e-3, seed=42):
    """Standardise features, fit the MLP on `device`, return (model, preds, stats)."""
    torch.manual_seed(seed)

    x_mean = x.mean(axis=0)
    x_std = x.std(axis=0)
    x_std[x_std == 0.0] = 1.0
    x_norm = (x - x_mean) / x_std

    x_t = torch.from_numpy(x_norm).float().to(device)
    y_t = torch.from_numpy(y).float().to(device)

    model = LossPredictor().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(x_t), y_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        preds_np = model(x_t).detach().cpu().numpy()

    feature_stats = {
        "x_mean_step": float(x_mean[0]), "x_mean_lr": float(x_mean[1]),
        "x_std_step":  float(x_std[0]),  "x_std_lr":  float(x_std[1]),
    }
    return model, preds_np, feature_stats


## Step 1 — Load env, connect to MLflow

In [7]:
from data_platform.common import set_env
from data_platform.tracking import MLflowTracking

set_env()
tracking = MLflowTracking.from_env()
tracking.check_connection()


[set_env] local: loaded /home/kiril/AI/BurnIT-BG/.env


True

## Step 2 — Define the run config

Note the extra GPU-specific params (`torch_version`, `cuda_*`). These show up
as searchable params in the UI so you can filter "show me only GPU runs".

In [8]:
total_steps = 600
warmup_steps = 60
eval_every = 50

run_tags = {"task": "pretraining-sim", "framework": "pytorch", "device": device.type}
run_params = {
    "total_steps": total_steps,
    "warmup_steps": warmup_steps,
    "eval_every": eval_every,
    "batch_tokens": 8192,
    "peak_lr": 3e-4,
    "min_lr": 1e-5,
    "optimizer": "adamw",
    "scheduler": "cosine",
    "primary_metric": "eval_f1",
    "torch_version": torch.__version__,
    "torch_device": device.type,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda or "n/a",
    "cuda_device_name": (
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a"
    ),
}


## Step 3 — Open the MLflow run

In [9]:
import mlflow
from pathlib import Path

NOTEBOOK_PATH = Path.cwd() / "example_training_plots_gpu.ipynb"

active_run = mlflow.start_run(
    run_name="llm-train-plots-gpu-example-nb",
    tags=run_tags,
    log_system_metrics=True,
)
print("active run id:", active_run.info.run_id)

tracking.log_hardware(step=0)
tracking.log_params(run_params)

if NOTEBOOK_PATH.exists():
    source_artifact_uri = tracking.save_data(NOTEBOOK_PATH, artifact_path=SOURCE_ARTIFACT_PATH)
    print("source artifact:", source_artifact_uri)


2026/05/10 21:47:30 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/05/10 21:47:30 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


active run id: d4b89a12dcb0411cb080357e06d23c4c


/home/kiril/AI/BurnIT-BG/venv/lib/python3.14/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


source artifact: runs:/d4b89a12dcb0411cb080357e06d23c4c/notebooks


## Step 4 — Simulate the training trajectory

In [10]:
with tracking.timed("simulate"):
    data = simulate_training(
        total_steps=total_steps,
        warmup_steps=warmup_steps,
        eval_every=eval_every,
    )

print("series keys:", list(data.keys()))


series keys: ['steps', 'train_loss', 'eval_steps', 'eval_loss', 'eval_precision', 'eval_recall', 'eval_f1', 'eval_accuracy', 'train_ppl', 'eval_ppl', 'learning_rate', 'grad_norm', 'tokens_per_sec', 'step_seconds']


Trace(trace_id=tr-50d63cf7ca799dfe51f8e378d7d53f9c)

## Step 5 — Build the history DataFrame and register as MLflow Dataset

In [11]:
import pandas as pd

history_df = pd.DataFrame({
    "step": data["steps"],
    "train_loss": data["train_loss"],
    "train_ppl": data["train_ppl"],
    "learning_rate": data["learning_rate"],
    "grad_norm": data["grad_norm"],
    "tokens_per_sec": data["tokens_per_sec"],
    "step_seconds": data["step_seconds"],
})

train_dataset = tracking.log_dataset(
    history_df, name="pretrain-history",
    targets="train_loss", context="training",
)
history_df.head()


/home/kiril/AI/BurnIT-BG/venv/lib/python3.14/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


,step,train_loss,train_ppl,learning_rate,grad_norm,tokens_per_sec,step_seconds
0,1,9.079604,8774.486370,0.000005,0.776666,6072.026692,1.349138
1,2,8.993383,8049.645934,0.000010,0.730662,6600.540650,1.241110
2,3,9.032720,8372.601639,0.000015,0.803660,6317.913518,1.296631
3,4,9.008193,8169.745176,0.000020,1.226472,6716.780073,1.219632
4,5,8.860545,7048.323216,0.000025,1.161212,6482.398456,1.263730


## Step 6 — Stream per-step metrics

In [12]:
import time

with tracking.timed("log_step_metrics") as streaming_timer:
    sim_start = time.perf_counter()
    for i, step in enumerate(data["steps"].tolist()):
        tracking.log_metrics({
            "train_loss":     float(data["train_loss"][i]),
            "train_ppl":      float(data["train_ppl"][i]),
            "learning_rate":  float(data["learning_rate"][i]),
            "grad_norm":      float(data["grad_norm"][i]),
            "tokens_per_sec": float(data["tokens_per_sec"][i]),
            "step_seconds":   float(data["step_seconds"][i]),
        }, step=int(step))
    for j, step in enumerate(data["eval_steps"].tolist()):
        tracking.log_metrics({
            "eval_loss":      float(data["eval_loss"][j]),
            "eval_ppl":       float(data["eval_ppl"][j]),
            "eval_accuracy":  float(data["eval_accuracy"][j]),
            "eval_precision": float(data["eval_precision"][j]),
            "eval_recall":    float(data["eval_recall"][j]),
            "eval_f1":        float(data["eval_f1"][j]),
        }, step=int(step))
    tracking.log_duration("step_logging_wallclock", time.perf_counter() - sim_start)

print(f"streamed in {streaming_timer.elapsed:.1f}s")


streamed in 52.6s


Trace(trace_id=tr-aa4378f87acf1ea9bbed61a849d986e0)

## Step 7 — Train the predictor on GPU

The actual GPU work happens here: features are standardised, moved to
`device`, and the MLP runs forward/backward/optimizer-step on CUDA for 400
epochs. With only 600 samples this finishes in milliseconds — the goal is to
exercise the GPU code path, not to actually need it.

In [13]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

with tracking.timed("fit_loss_predictor"):
    x_train = history_df[["step", "learning_rate"]].to_numpy()
    y_train = history_df["train_loss"].to_numpy()

    loss_predictor, preds, feature_stats = train_loss_predictor(
        x_train, y_train, device=device,
    )

    tracking.log_params({
        "predictor_kind": "MLP",
        "predictor_features": "step,learning_rate",
        "predictor_hidden": 32,
        "predictor_epochs": 400,
        "predictor_lr": 5e-3,
        "predictor_optimizer": "adam",
        **feature_stats,
    })
    tracking.log_metrics({
        "predictor_mae": float(mean_absolute_error(y_train, preds)),
        "predictor_r2":  float(r2_score(y_train, preds)),
    }, dataset=train_dataset)

print("MLP MAE:", mean_absolute_error(y_train, preds))
print("MLP R^2:", r2_score(y_train, preds))


MLP MAE: 0.04223753128703149
MLP R^2: 0.9992255283694332


Trace(trace_id=tr-dde7e5b9a5a082d7bc29d92d182bc776)

## Step 8 — Log AND register the PyTorch model

A few GPU-specific points:

* `loss_predictor.cpu()` — `mlflow.pytorch.log_model` serialises the
  `state_dict`, but the input example tensor must live on CPU.
* `flavor="pytorch"` routes the call to `mlflow.pytorch.log_model` (different
  on-disk format than the sklearn flavor).
* `input_example` must be already standardised — the loaded model expects
  normalised inputs, so we apply the same mean/std used at training time.

In [14]:
x_mean = np.array([feature_stats["x_mean_step"], feature_stats["x_mean_lr"]])
x_std  = np.array([feature_stats["x_std_step"],  feature_stats["x_std_lr"]])
input_example = ((x_train[:3] - x_mean) / x_std).astype(np.float32)

with tracking.timed("log_register_model"):
    tracking.log_model(
        loss_predictor.cpu(),
        flavor="pytorch",
        artifact_path=MODEL_ARTIFACT_PATH,
        input_example=input_example,
        params={"hidden": 32, "features": ["step", "learning_rate"], "trained_on": device.type},
        registered_model_name=REGISTERED_MODEL_NAME,
        allow_registry_failure=True,
    )

print(f"logged + registered as: {REGISTERED_MODEL_NAME}")


2026/05/10 21:48:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/05/10 21:48:31 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/10 21:48:34 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version la

logged + registered as: burnit-bg-loss-predictor-gpu


Trace(trace_id=tr-456dbce07f6dfb3595db0f1586d54d49)

## Step 9 — Build and log the plot bundle

(Plot code is identical to the CPU notebook — same simulated metrics drive
the same plots.)

In [15]:
from utils.plots import (
    plot_attention_heatmap, plot_eval_benchmarks, plot_gradient_norms,
    plot_learning_rate_schedule, plot_loss_curves, plot_perplexity,
    plot_step_time, plot_throughput, plot_token_length_distribution,
    plot_training_dashboard,
)


In [16]:
with tracking.timed("plot_and_log"):
    with tracking.trace("plot_loss"):
        fig, _ = plot_loss_curves(
            train_loss=data["train_loss"], eval_loss=data["eval_loss"],
            steps=data["steps"], eval_steps=data["eval_steps"],
            title="Train vs Eval Loss",
        )
        tracking.log_plot(fig, key="loss_curves")

    with tracking.trace("plot_perplexity"):
        fig, _ = plot_perplexity(
            train_ppl=data["train_ppl"], eval_ppl=data["eval_ppl"],
            steps=data["steps"], eval_steps=data["eval_steps"],
        )
        tracking.log_plot(fig, key="perplexity")

    with tracking.trace("plot_lr"):
        fig, _ = plot_learning_rate_schedule(
            learning_rates=data["learning_rate"], steps=data["steps"],
        )
        tracking.log_plot(fig, key="learning_rate")

    with tracking.trace("plot_grad"):
        fig, _ = plot_gradient_norms(
            grad_norms=data["grad_norm"], steps=data["steps"], clip_threshold=1.0,
        )
        tracking.log_plot(fig, key="gradient_norms")

    with tracking.trace("plot_throughput"):
        fig, _ = plot_throughput(
            tokens_per_sec=data["tokens_per_sec"], steps=data["steps"],
        )
        tracking.log_plot(fig, key="throughput")

    with tracking.trace("plot_step_time"):
        fig, _ = plot_step_time(step_seconds=data["step_seconds"], steps=data["steps"])
        tracking.log_plot(fig, key="step_time")

    with tracking.trace("plot_dashboard"):
        fig, _ = plot_training_dashboard(
            history_df.drop(columns=["train_ppl"]),
            title="Training Dashboard",
        )
        tracking.log_plot(fig, key="training_dashboard")

    with tracking.trace("plot_token_lengths"):
        rng = np.random.default_rng(0)
        lengths = rng.integers(low=8, high=2048, size=20_000, endpoint=True)
        fig, _ = plot_token_length_distribution(lengths)
        tracking.log_plot(fig, key="token_length_distribution")

    with tracking.trace("plot_attention"):
        tokens = list("ABCDEFGHIJKLMNOP")
        fig, _ = plot_attention_heatmap(
            fake_attention_matrix(n_tokens=len(tokens)),
            tokens=tokens, title="Attention (synthetic)",
        )
        tracking.log_plot(fig, key="attention")

    with tracking.trace("plot_benchmarks"):
        scores = {"winogrande": 0.61, "arc_easy": 0.55, "hellaswag": 0.42,
                  "lambada": 0.38, "piqa": 0.66}
        baseline = {k: v - 0.04 for k, v in scores.items()}
        fig, _ = plot_eval_benchmarks(scores, baseline=baseline)
        tracking.log_plot(fig, key="eval_benchmarks")

print("plots uploaded")


plots uploaded


Trace(trace_id=tr-13daf2d0e0b92dfd37ff09d014a32c64)

## Step 10 — Final summary metrics + hardware snapshot

In [17]:
tracking.log_metrics({
    "final_train_loss":    float(data["train_loss"][-1]),
    "final_eval_loss":     float(data["eval_loss"][-1]),
    "final_train_ppl":     float(data["train_ppl"][-1]),
    "final_eval_ppl":      float(data["eval_ppl"][-1]),
    "final_eval_f1":       float(data["eval_f1"][-1]),
    "best_eval_f1":        float(np.max(data["eval_f1"])),
    "mean_tokens_per_sec": float(np.mean(data["tokens_per_sec"])),
    "total_train_seconds": float(np.sum(data["step_seconds"])),
}, dataset=train_dataset)

tracking.log_hardware(step=1)


## Step 11 — Close the run

In [18]:
mlflow.end_run()
print(f"run finished. registered model: {REGISTERED_MODEL_NAME} (alias: {MODEL_ALIAS})")
print(f"device used for predictor: {device.type}")


2026/05/10 21:48:46 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...


🏃 View run llm-train-plots-gpu-example-nb at: https://k3s-acer-f5-master.tail1e4f6a.ts.net/mlflow/#/experiments/3/runs/d4b89a12dcb0411cb080357e06d23c4c
🧪 View experiment at: https://k3s-acer-f5-master.tail1e4f6a.ts.net/mlflow/#/experiments/3


2026/05/10 21:48:46 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


run finished. registered model: burnit-bg-loss-predictor-gpu (alias: staging)
device used for predictor: cuda


## Notes on the warnings you'll see

When you run the model-logging cell, MLflow may emit a few warnings:

1. **`Saving pytorch model by Pickle ...`** — MLflow defaults to pickle for
   `nn.Module`. It's a security advisory, not an error. Switch to the
   safer `pt2` format by passing `serialization_format="pt2"` to
   `mlflow.pytorch.log_model` if you need it.

2. **`Found torch version 2.x.y+cuNNN ...`** — your local torch is built
   against a specific CUDA wheel; MLflow strips the `+cuNNN` tag because
   PyPI doesn't host CUDA-tagged wheels. Effect: anyone reloading the model
   in a fresh env via auto-installed deps will get the **CPU** torch wheel.
   For inference of this tiny MLP that's fine.

3. **`Skip logging GPU metrics`** — MLflow's system-metrics monitor needs
   `pynvml`. `pip install pynvml` to enable the live GPU utilisation /
   VRAM charts on the run page. (Your `log_hardware()` snapshots from
   `nvidia-smi` are independent and still recorded.)
